In [5]:
import pandas as pd
from dqn.dqn_config import HYPER_PARAMS
import os
from colorama import Fore
pd.set_option('display.width', 2000)              # increase total line width
pd.set_option('display.expand_frame_repr', False) # disable wrapping into multiple rows

In [6]:
# Directory containing the CSV files
DATA_DIR = ["test_seed_train","test_seed_test"]

# Map filenames to model names
file_model_map = {
    "transition_csv_DoubleDQN_OnRL.csv":"On-DDQN",
    "transition_csv_DoubleDQN_OffRL4e6.csv":"Off-DDQN-4e6",
    "transition_csv_BCQ_BCQ4e6.csv":"BCQ-4e6",
    "transition_csv_DoubleDQN_OffRL4e6Clean.csv":"Off-DDQN-4e6-Clean",
    "transition_csv_BCQ_BCQ4e6Clean.csv":"BCQ-4e6-Clean",
    "transition_csv_DoubleDQN_OffRL4e6Shuffle.csv":"Off-DDQN-4e6-Shuffle",
    "transition_csv_BCQ_BCQ4e6Shuffle.csv":"BCQ-4e6-Shuffle",
    "transition_csv_DoubleDQN_OffRL4e6CleanShuffle.csv":"Off-DDQN-4e6-Clean-Shuffle",
    "transition_csv_BCQ_BCQ4e6CleanShuffle.csv":"BCQ-4e6-Clean-Shuffle",
    "transition_csv_DoubleDQN_OffRL8e6Clean.csv":"Off-DDQN-8e6-Clean",
    "transition_csv_BCQ_BCQ8e6Clean.csv":"BCQ-8e6-Clean",
}

DIR_CLEAN = os.path.join(HYPER_PARAMS.agent_data_dir,"train",'transition_csv_DoubleDQN_OnRL_16e6_cleaned')

csv_files_clean=[os.path.join(DIR_CLEAN, f) for f in os.listdir(DIR_CLEAN)  if f.endswith('.csv')]

# Read and combine all CSV files into one DataFrame
df_clean = pd.concat([pd.read_csv(f) for f in csv_files_clean], ignore_index=True)

state_cols_name=df_clean.columns[1:18]
trans_cols_name=[c for c in df_clean.columns if c != "trans_idx"]

unique_train_states=set(map(tuple, df_clean[state_cols_name].to_numpy()))
unique_train_trans=set( map(tuple, df_clean[trans_cols_name].to_numpy()))

## Unseen State and Transition Analysis of Testing Dataset

In [7]:
for DIR in DATA_DIR:
    records = []

    print(Fore.YELLOW+ f"\n{'='*15}\n"+("Training Seeds" if DIR == "test_seed_train" else "Testing Seeds")+  f"\n{'='*15}",Fore.RESET)

    for file_name, model_name in file_model_map.items():
        
        filepath = os.path.join(HYPER_PARAMS.agent_data_dir, DIR, file_name)

    
        df = pd.read_csv(filepath)
        unique_states = set(map(tuple, df[state_cols_name].to_numpy()))
        unseen_unique_states = unique_states - unique_train_states
        all_states = list(map(tuple, df[state_cols_name].to_numpy()))
        num_unseen_state= sum(
            state not in unique_train_states
            for state in all_states
        )

        unique_trans= set(map(tuple, df[trans_cols_name].to_numpy()))
        unseen_unique_trans = unique_trans - unique_train_trans
        all_trans = list(map(tuple, df[trans_cols_name].to_numpy()))
        num_unseen_trans= sum(
            trans not in unique_train_trans
            for trans in all_trans
        )

        record = {
            "Model": model_name,
            "Number of Steps": len(df),
            "Unique States": len(unique_states),
            "Unseen Unique States": len(unseen_unique_states),
            "Unseen Unique State (%)":str(round(len(unseen_unique_states)/len(unique_states)*100,2))+"%",
            "Unseen State (%)":str(round(num_unseen_state/len(all_states)*100,2))+"%",

            "Unique Transitions": len(unique_trans),
            "Unseen Unique Transitions": len(unseen_unique_trans),
            "Unseen Unique Transition (%)":str(round(len(unseen_unique_trans)/len(unique_trans)*100,2))+"%",
            "Unseen Transition (%)":str(round(num_unseen_trans/len(all_trans)*100,2))+"%",

        }

        records.append(record)
        # print(f"✓ Processed: {model_name}")

    output_path = "./test_data_analyse_"+DIR[5:]+".csv"
    

    summary_df = pd.DataFrame(records).set_index("Model")
    # summary_df = summary_df.round(2)
    summary_df = summary_df.T
    summary_df.index.name = "Metric"

    print(summary_df, "\n")

    summary_df.to_csv(output_path)


Training Seeds
Model                        On-DDQN Off-DDQN-4e6 BCQ-4e6 Off-DDQN-4e6-Clean BCQ-4e6-Clean Off-DDQN-4e6-Shuffle BCQ-4e6-Shuffle Off-DDQN-4e6-Clean-Shuffle BCQ-4e6-Clean-Shuffle Off-DDQN-8e6-Clean BCQ-8e6-Clean
Metric                                                                                                                                                                                                           
Number of Steps                 1422         1441    1421               1439          1427                 1435            1447                       1464                  1438               1447          1442
Unique States                   1301         1296    1252               1310          1283                 1379            1357                       1266                  1286               1219          1248
Unseen Unique States             767          827     693                762           688                 1068             983                 